In [1]:
import pandas as pd

df = pd.read_csv("/content/simulated_household_energy_20000.csv")
print(df.shape)
df.head()

(20000, 18)


,household_id,house_size,sqft,occupants,owns_PhoneCharging,owns_Radio,owns_TV,owns_IronBox,owns_ElectricKettle,owns_Fridge,owns_Microwave,owns_WashingMachine,owns_Laptop,owns_DeepFreezer,owns_Fan,owns_WaterHeater,monthly_consumption_kwh,monthly_bill_kes
0,H00001,2 Beds,857.0,5,1,1,1,1,1,1,1,0,0,0.0,0,0,164.34,4139
1,H00002,3+ Beds,1219.0,6,0,0,1,1,0,1,0,0,1,0.0,1,0,182.28,4640
2,H00003,2 Beds,892.0,4,1,1,1,1,0,0,0,0,1,0.0,1,0,97.47,2278
3,H00004,1 Bed,629.0,3,1,1,1,1,0,0,1,0,0,0.0,0,1,124.06,3013
4,H00005,1 Bed,676.0,2,1,0,0,1,1,1,0,0,0,0.0,0,1,117.53,2831


In [2]:
# Blank deep-freezer values: treat as "does not own" (about 90% of homes don't)
df["owns_DeepFreezer"] = df["owns_DeepFreezer"].fillna(0).astype(int)

# Drop columns the model should not use
df = df.drop(columns=["household_id", "sqft", "monthly_bill_kes"])

print(df.isnull().sum().sum())   # should print 0
print(df.shape)

0
(20000, 15)


In [3]:
from sklearn.model_selection import train_test_split

# Turn house_size text into 0/1 columns the model can read
X = pd.get_dummies(df.drop(columns=["monthly_consumption_kwh"]),
                   columns=["house_size"], dtype=int)
y = df["monthly_consumption_kwh"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)
print(list(X.columns))

(16000, 17) (4000, 17)
['occupants', 'owns_PhoneCharging', 'owns_Radio', 'owns_TV', 'owns_IronBox', 'owns_ElectricKettle', 'owns_Fridge', 'owns_Microwave', 'owns_WashingMachine', 'owns_Laptop', 'owns_DeepFreezer', 'owns_Fan', 'owns_WaterHeater', 'house_size_1 Bed', 'house_size_2 Beds', 'house_size_3+ Beds', 'house_size_Studio']


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                           random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "Model": name,
        "MAE (kWh)": round(mean_absolute_error(y_test, pred), 2),
        "R2": round(r2_score(y_test, pred), 4),
    })

pd.DataFrame(results).sort_values("MAE (kWh)")

,Model,MAE (kWh),R2
0,Linear Regression,11.42,0.8842
2,Gradient Boosting,11.57,0.8806
1,Random Forest,11.99,0.8721


In [5]:
best = models["Linear Regression"]

weights = pd.Series(best.coef_, index=X.columns).sort_values(ascending=False)
print("Base:", round(best.intercept_, 2))
weights.round(2)

Base: 30.95


,0
owns_DeepFreezer,57.06
owns_WaterHeater,44.44
owns_Fridge,42.79
house_size_3+ Beds,14.85
owns_TV,12.16
owns_Laptop,9.44
owns_ElectricKettle,9.29
owns_Microwave,9.25
occupants,8.75
owns_IronBox,8.74


In [6]:
import joblib

joblib.dump(best, "energy_model.joblib")
joblib.dump(list(X.columns), "model_columns.joblib")

['model_columns.joblib']